# FinBERT scoring for Polygon + market-context news


In [ ]:
!pip -q install transformers torch pandas tqdm

In [ ]:
from google.colab import files
uploaded = files.upload()
INPUT = 'news_target_tickers_2022_2023_polygon_market_quality.csv'
assert INPUT in uploaded, f'Upload {INPUT}'

In [ ]:
import pandas as pd

news = pd.read_csv(INPUT)
news['article_id'] = news['article_id'].astype(str)
news['text'] = news['text'].fillna('').astype(str)
articles = news[['article_id', 'published_utc', 'text']].drop_duplicates('article_id').reset_index(drop=True)
print('article-ticker rows:', len(news))
print('unique articles to score:', len(articles))
print(news.groupby('ticker').agg(rows=('ticker', 'size'), unique_articles=('article_id', 'nunique'), news_days=('date', 'nunique')))

In [ ]:
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = 'ProsusAI/finbert'
BATCH_SIZE = 64
MAX_LENGTH = 512

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()
id2label = {int(k): str(v).lower() for k, v in model.config.id2label.items()}
print(id2label)

In [ ]:
def canonical_label(label):
    label = str(label).lower().strip()
    return {'pos': 'positive', 'neg': 'negative'}.get(label, label)

rows = []
for start in tqdm(range(0, len(articles), BATCH_SIZE)):
    batch = articles.iloc[start:start + BATCH_SIZE]
    encoded = tokenizer(
        batch['text'].tolist(),
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors='pt',
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        probs = torch.softmax(model(**encoded).logits, dim=1).detach().cpu().numpy()
    for (_, item), prob in zip(batch.iterrows(), probs):
        scores = {'positive': 0.0, 'negative': 0.0, 'neutral': 0.0}
        for idx, score in enumerate(prob):
            label = canonical_label(id2label[idx])
            scores[label] = float(score)
        rows.append({
            'article_id': item['article_id'],
            'published_utc': item['published_utc'],
            'text': item['text'],
            'finbert_positive': scores['positive'],
            'finbert_negative': scores['negative'],
            'finbert_neutral': scores['neutral'],
            'finbert_sentiment_score': scores['positive'] - scores['negative'],
        })

scores = pd.DataFrame(rows)
scores['finbert_predicted_label'] = scores[['finbert_positive', 'finbert_negative', 'finbert_neutral']].idxmax(axis=1).str.replace('finbert_', '', regex=False)
OUTPUT = 'finbert_article_scores_2022_2023_polygon_market_quality.csv'
scores.to_csv(OUTPUT, index=False)
print(scores[['finbert_positive', 'finbert_negative', 'finbert_neutral']].describe())
print(scores['finbert_predicted_label'].value_counts(normalize=True))
files.download(OUTPUT)